# เทรนโมเดลตรวจสอบฟอร์มออกกำลังกาย (Exercise Form Classifier)

Notebook นี้ใช้สำหรับเทรนโมเดลจำแนกฟอร์มท่าออกกำลังกาย (ถูก/ผิด) จากชุดข้อมูลภาพ 10,000+ รูป
โดยมีขั้นตอนหลัก:
1. ติดตั้งไลบรารีและเชื่อมต่อ Google Drive
2. แตกจุด landmark ร่างกาย (33 จุด) จากภาพด้วย MediaPipe Pose
3. แปลง landmark เป็น feature vector (มุมข้อต่อ + ตำแหน่งจุด)
4. เทรนโมเดล Neural Network จำแนกคลาส (เช่น correct_squat, knee_too_far, back_not_straight)
5. Export เป็น .h5 และ .tflite เพื่อนำไปใช้ใน FastAPI backend

**โครงสร้างชุดข้อมูลที่คาดหวัง (เก็บใน Google Drive):**
```
dataset/
  squat/
    correct/        (รูปท่า squat ที่ถูกต้อง)
    knee_too_far/
    back_not_straight/
  pushup/
    correct/
    hip_sagging/
    elbow_too_wide/
  plank/
    correct/
    hip_too_high/
    hip_too_low/
```
รวมทุกคลาสควรมีรูปภาพอย่างน้อย 10,000 รูป เพื่อให้โมเดล generalize ได้ดี

## 1. ติดตั้งไลบรารีและเชื่อมต่อ Google Drive

In [ ]:
!pip install -q mediapipe opencv-python-headless tensorflow scikit-learn

from google.colab import drive
drive.mount('/content/drive')

DATASET_PATH = '/content/drive/MyDrive/fitness_dataset'  # แก้ path ให้ตรงกับของจริง
MODEL_OUTPUT_PATH = '/content/drive/MyDrive/fitness_models'

import os
os.makedirs(MODEL_OUTPUT_PATH, exist_ok=True)

## 2. ฟังก์ชันแตก Landmark จากภาพด้วย MediaPipe Pose

In [ ]:
import cv2
import numpy as np
import mediapipe as mp
import glob
from tqdm import tqdm

mp_pose = mp.solutions.pose

def extract_landmark_vector(image_path, pose_model):
    """อ่านภาพ -> ตรวจจับ pose -> คืนค่า feature vector (33 จุด x 3 ค่า = 99 มิติ)"""
    image = cv2.imread(image_path)
    if image is None:
        return None
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    results = pose_model.process(image_rgb)
    if not results.pose_landmarks:
        return None
    vec = []
    for lm in results.pose_landmarks.landmark:
        vec.extend([lm.x, lm.y, lm.visibility])
    return np.array(vec, dtype=np.float32)

## 3. โหลดข้อมูลทั้งหมด (10,000+ รูป) แล้วแปลงเป็น Dataset

In [ ]:
EXERCISE_TYPE = 'squat'  # เปลี่ยนเป็น 'pushup' หรือ 'plank' เพื่อเทรนทีละชนิดท่า
class_dirs = sorted(os.listdir(f'{DATASET_PATH}/{EXERCISE_TYPE}'))
label_map = {name: idx for idx, name in enumerate(class_dirs)}
print('คลาสที่พบ:', label_map)

X, y = [], []
with mp_pose.Pose(static_image_mode=True, model_complexity=2, min_detection_confidence=0.5) as pose_model:
    for class_name in class_dirs:
        image_paths = glob.glob(f'{DATASET_PATH}/{EXERCISE_TYPE}/{class_name}/*.jpg') + \
                      glob.glob(f'{DATASET_PATH}/{EXERCISE_TYPE}/{class_name}/*.png')
        print(f'{class_name}: {len(image_paths)} รูป')
        for img_path in tqdm(image_paths, desc=class_name):
            vec = extract_landmark_vector(img_path, pose_model)
            if vec is not None:
                X.append(vec)
                y.append(label_map[class_name])

X = np.array(X)
y = np.array(y)
print('จำนวนข้อมูลทั้งหมดที่แตก landmark ได้สำเร็จ:', X.shape, y.shape)

## 4. แบ่งข้อมูล Train/Validation/Test

In [ ]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

num_classes = len(label_map)
y_cat = to_categorical(y, num_classes=num_classes)

X_train, X_temp, y_train, y_temp = train_test_split(X, y_cat, test_size=0.3, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print('Train:', X_train.shape, 'Val:', X_val.shape, 'Test:', X_test.shape)

## 5. สร้างและเทรนโมเดล (Dense Neural Network)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

def build_model(input_dim, num_classes):
    model = models.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(64, activation='relu'),
        layers.Dense(num_classes, activation='softmax'),
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss='categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model

model = build_model(X_train.shape[1], num_classes)
model.summary()

early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
reduce_lr = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=64,
    callbacks=[early_stop, reduce_lr],
)

## 6. ประเมินผลบนชุด Test

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f'Test Accuracy: {test_acc*100:.2f}%')

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['accuracy'], label='train')
axes[0].plot(history.history['val_accuracy'], label='val')
axes[0].set_title('Accuracy'); axes[0].legend()
axes[1].plot(history.history['loss'], label='train')
axes[1].plot(history.history['val_loss'], label='val')
axes[1].set_title('Loss'); axes[1].legend()
plt.show()

## 7. บันทึกโมเดลเป็น .h5 และแปลงเป็น .tflite (เบาและเร็วกว่า สำหรับ deploy)

In [ ]:
h5_path = f'{MODEL_OUTPUT_PATH}/{EXERCISE_TYPE}_form_classifier.h5'
model.save(h5_path)
print('บันทึกโมเดล .h5 ที่:', h5_path)

import json
with open(f'{MODEL_OUTPUT_PATH}/{EXERCISE_TYPE}_label_map.json', 'w') as f:
    json.dump(label_map, f, ensure_ascii=False, indent=2)

converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()
tflite_path = f'{MODEL_OUTPUT_PATH}/{EXERCISE_TYPE}_form_classifier.tflite'
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)
print('บันทึกโมเดล .tflite ที่:', tflite_path)

## 8. นำไปใช้งานจริง

ดาวน์โหลดไฟล์ `{EXERCISE_TYPE}_form_classifier.h5` จาก Google Drive แล้ววางไว้ที่:
```
backend/app/ml_models/exercise_form_classifier.h5
```
ระบบ FastAPI (`app/utils/pose_processing.py` ฟังก์ชัน `load_custom_model`) จะโหลดไฟล์นี้โดยอัตโนมัติ
และใช้ทำนายความถูกต้องของฟอร์มแทนการประเมินแบบ rule-based